# Lab 2 – Building Guardrail Systems

**Estimated Time:** 100–130 minutes  
**Difficulty:** Intermediate

## Learning Objectives
- Implement layered guardrails for LLM applications
- Detect and redact PII before requests reach foundation models
- Build validation pipelines combining rule-based and ML-based checks
- Orchestrate guardrail execution for both inputs and outputs
- Monitor guardrail performance and handle compliance reporting

## Prerequisites & Setup
- Python 3.10+ environment with `guardrails-ai`, `presidio-analyzer`, `presidio-anonymizer`, `transformers`, `torch`, `langfuse`, `httpx`, `pandas`, `numpy` installed
- Environment variables set for external APIs as needed: `OPENAI_API_KEY`, `PERSPECTIVE_API_KEY` (optional), and access tokens for any moderation services used
- Download required spaCy or stanza models for Presidio if prompted
- Optional: Hugging Face cache warmed for toxicity detection models

> **Note:** Some guardrail checks call remote services. Mock or stub the calls if working offline.

## Lab Structure
You will complete eight exercises that progressively assemble a production-grade guardrail stack.

1. Content moderation adapters
2. PII detection and redaction pipelines
3. Custom rule engine for business policies
4. Structured output validation
5. Toxicity detection with transformers
6. Guardrail orchestration (input/output)
7. Performance monitoring and caching
8. Compliance reporting and audit logging

Each exercise includes scaffolding with `TODO` markers. Fill in the required logic, execute the cell, and capture your observations in the reflection prompts.

## Exercise 1: Build Content Moderation Adapters
**Scenario:** Incoming prompts must be screened using both the OpenAI Moderation API and a custom blocklist.

**Success Criteria**
- OpenAI Moderation API errors handled gracefully with fallback
- Custom regex blocklist evaluated before external calls
- Unified response structure with fields `flagged`, `severity`, `reasons`

**Hints**
- Wrap API calls in try/except to isolate transient failures
- Capture latency for observability
- Normalize categories (e.g., `self-harm` → `SELF_HARM`) for downstream systems

In [ ]:
import os
import re
import time
from typing import Dict, List

from openai import OpenAI

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
BLOCKLIST_PATTERNS = [
    re.compile(r"\b(make|build)\s+(?:a|the)\s+bomb\b", re.IGNORECASE),
    # TODO: Add more domain-specific patterns as needed
]

def check_blocklist(text: str) -> List[str]:
    matched = []
    for pattern in BLOCKLIST_PATTERNS:
        if pattern.search(text):
            matched.append(pattern.pattern)
    return matched

def moderate_with_openai(text: str) -> Dict:
    start = time.perf_counter()
    try:
        # TODO: Call OpenAI moderation endpoint and capture response
        # TODO: Normalize categories and include latency
        ...
    except Exception as exc:
        # TODO: Return a fallback structure with error context and set flagged=False
        ...

def moderate_content(text: str) -> Dict:
    



    reasons = []
    blocklist_matches = check_blocklist(text)
    if blocklist_matches:
        reasons.extend([f"BLOCKLIST:{pattern}" for pattern in blocklist_matches])
    # TODO: Short-circuit if blocklist already flagged content
    # TODO: Merge OpenAI moderation result (if executed) into unified response
    ...

# TODO: Smoke test with a harmless prompt and a malicious prompt once implemented

**Reflection & Verification**
- [ ] Fallback logic prevents lab execution from failing without OpenAI connectivity
- [ ] Combined result clearly communicates severity and reasons
- [ ] Latency metrics captured for both blocklist and API checks

## Exercise 2: Implement PII Detection & Redaction Pipeline
**Scenario:** Detect sensitive information in user input and redact before LLM invocation.

**Success Criteria**
- Presidio analyzer detects default PII entities and returns structured metadata
- Anonymizer replaces detected entities with type-specific tokens (e.g., `[PERSON]`)
- Custom regex handles domain identifiers not covered by Presidio

**Hints**
- Cache the analyzer/anonymizer to avoid re-instantiation cost
- Combine Presidio results with custom matches before anonymization
- Include confidence scores in the returned payload

In [ ]:
from typing import Any, Tuple

from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

PII_ANALYZER = AnalyzerEngine()
PII_ANONYMIZER = AnonymizerEngine()
CUSTOM_ID_REGEX = re.compile(r"\bACCT-[0-9]{6}\b")

def detect_custom_ids(text: str) -> List[Dict[str, Any]]:
    return [
        {"entity_type": "ACCOUNT_ID", "start": match.start(), "end": match.end(), "score": 0.95}
        for match in CUSTOM_ID_REGEX.finditer(text)
    ]

def detect_pii(text: str) -> List[Dict[str, Any]]:
    # TODO: Call Presidio analyzer and merge with custom ID detections
    ...

def redact_pii(text: str, detections: List[Dict[str, Any]]) -> Tuple[str, List[Dict[str, Any]]]:
    # TODO: Build Presidio-style results and configure type-aware replacements
    # TODO: Return redacted text plus original detections
    ...

# TODO: Write a quick demo showing detection of email + custom account id

**Reflection & Verification**
- [ ] Multiple entity types redacted correctly with labels
- [ ] Confidence scores preserved for analytics
- [ ] Edge cases (e.g., overlapping entities) handled gracefully

## Exercise 3: Design a Custom Rule Engine
**Scenario:** Your compliance team needs business-specific guardrails (e.g., no financial advice for retail users).

**Success Criteria**
- Rule definitions stored as dataclasses with conditions
- Engine evaluates rules against request context and returns violations
- Support conditional severity (warning vs. block)

**Hints**
- Use callable predicates to evaluate context
- Provide override mechanisms for approved power users
- Include helpful remediation guidance in violation messages

In [ ]:
from dataclasses import dataclass
from typing import Callable, NamedTuple

class RuleViolation(NamedTuple):
    rule_name: str
    severity: str
    message: str

@dataclass
class GuardrailRule:
    name: str
    severity: str
    predicate: Callable[[Dict[str, Any]], bool]
    remediation: str

class RuleEngine:
    def __init__(self, rules: List[GuardrailRule]):
        self.rules = rules

    def evaluate(self, context: Dict[str, Any]) -> List[RuleViolation]:
        violations = []
        for rule in self.rules:
            # TODO: Skip evaluation for users with override flag
            # TODO: Invoke predicate and append violations with remediation hints
            ...
        return violations

# TODO: Define sample rules (no_financial_advice, restrict_beta_feature) and test evaluation

**Reflection & Verification**
- [ ] Rule engine differentiates warnings vs. blocking violations
- [ ] Overrides allow trusted users to bypass specific rules
- [ ] Remediation messages provide actionable guidance

In [ ]:
from guardrails import Guard
from guardrails.validators import ValidLength, ValidRange
from pydantic import BaseModel, Field, validator

class SupportResponse(BaseModel):
    summary: str = Field(..., min_length=20, max_length=400)
    sentiment: str = Field(..., regex=r"^(positive|neutral|negative)$")
    confidence: float = Field(..., ge=0.0, le=1.0)
    action_items: List[str] = Field(default_factory=list)

    @validator('action_items')
    def limit_action_items(cls, value: List[str]) -> List[str]:
        # TODO: Enforce at most 5 action items and trim whitespace
        ...
        return value

support_guard = Guard.from_pydantic(
    output_class=SupportResponse,
    prompt="Generate a support response JSON object...",
)

def run_structured_completion(prompt: str) -> SupportResponse:
    # TODO: Call your LLM client and pass raw output into guard.parse
    # TODO: Implement re-ask logic (num_reasks=2) and surface validation errors
    ...

# TODO: Simulate a failure by providing malformed JSON and confirm auto-correction

**Reflection & Verification**
- [ ] Invalid responses trigger re-ask attempts before failing
- [ ] Actionable error messages logged for debugging
- [ ] Final validated output adheres to schema

## Exercise 5: Integrate Toxicity Detection Models
**Scenario:** Augment guardrails with transformer-based toxicity scores.

**Success Criteria**
- Pipeline loads (or lazily loads) a Hugging Face toxicity model
- Function returns detailed scores and flags text above threshold
- Batch scoring supported for throughput

**Hints**
- Use `device_map="auto"` where available to leverage GPUs
- Cache pipeline instance to avoid reloading on each call
- Consider calibrating thresholds per category

In [ ]:
from functools import lru_cache
from transformers import pipeline

@lru_cache(maxsize=1)
def get_toxicity_pipeline(model_name: str = "unitary/toxic-bert"):
    # TODO: Initialize and return a Hugging Face text classification pipeline
    ...

def score_toxicity(texts: List[str], threshold: float = 0.7) -> List[Dict[str, Any]]:
    classifier = get_toxicity_pipeline()
    # TODO: Run classifier, parse scores per label, and flag toxicity
    ...

# TODO: Evaluate harmless and toxic samples to validate behaviour

**Reflection & Verification**
- [ ] Model loads only once per session
- [ ] Output includes raw scores for analytics and a boolean flag for enforcement
- [ ] Threshold tuning documented based on experiments

## Exercise 6: Orchestrate Guardrail Execution
**Scenario:** Compose input and output guardrails into a single pipeline with audit logging.

**Success Criteria**
- Pipeline executes content moderation, PII redaction, rule engine, and toxicity checks
- Guardrail results recorded with timestamps and correlation IDs
- Execution short-circuits on blocking violations

**Hints**
- Reuse components from previous exercises
- Allow configurable execution order for experimentation
- Produce a structured report summarizing all checks

In [ ]:
import uuid
from datetime import datetime

class GuardrailReport(NamedTuple):
    correlation_id: str
    steps: List[Dict[str, Any]]
    blocked: bool
    message: str

class GuardrailPipeline:
    def __init__(self):
        self.audit_log: List[Dict[str, Any]] = []

    def run(self, text: str, context: Dict[str, Any]) -> GuardrailReport:
        correlation_id = str(uuid.uuid4())
        steps: List[Dict[str, Any]] = []
        blocked = False
        message = ""

        def record_step(name: str, result: Dict[str, Any]):
            steps.append({
                "name": name,
                "timestamp": datetime.utcnow().isoformat(),
                "result": result,
            })

        # TODO: Execute moderation guardrail and record outcome
        # TODO: Execute PII pipeline (redaction) and update text if needed
        # TODO: Evaluate business rules and halt on blocking violations
        # TODO: Run toxicity scoring on sanitized output

        report = GuardrailReport(correlation_id, steps, blocked, message)
        self.audit_log.append(report._asdict())
        return report

# TODO: Instantiate pipeline, run with sample inputs, and examine audit log

**Reflection & Verification**
- [ ] Pipeline halts on blocking violations while logging all prior steps
- [ ] Reports include correlation IDs for traceability
- [ ] Sanitized text returned for downstream LLM calls

## Exercise 7: Optimize Guardrail Performance
**Scenario:** Guardrail checks must operate within tight latency budgets at scale.

**Success Criteria**
- Implement caching for deterministic checks (e.g., blocklist results)
- Parallelize compatible tasks with `asyncio.gather`
- Collect execution timing metrics and export via Langfuse or logging

**Hints**
- Use `functools.lru_cache` for pure functions
- Wrap guardrail steps with timers to measure duration
- Batch remote API calls when possible

In [ ]:
import asyncio
from functools import lru_cache

@lru_cache(maxsize=1024)
def cached_blocklist_check(text_hash: str) -> List[str]:
    # TODO: Map hash back to text or pre-hash before calling
    ...

async def run_parallel_checks(text: str) -> Dict[str, Any]:
    # TODO: Kick off moderation + toxicity checks concurrently
    ...

def time_guardrail_step(name: str, func, *args, **kwargs) -> Dict[str, Any]:
    # TODO: Measure execution time, capture outcome, and emit metrics
    ...

# TODO: Integrate these helpers into GuardrailPipeline once complete

**Reflection & Verification**
- [ ] Caching reduces latency for repeated prompts
- [ ] Parallel execution respects dependency ordering
- [ ] Timing metrics feed into observability dashboards

## Exercise 8: Generate Compliance Reports
**Scenario:** Legal requires weekly reports summarizing guardrail activity (PII removal, violations, overrides).

**Success Criteria**
- Aggregate audit logs into reportable metrics (counts, severities, overrides)
- Produce structured output for downstream storage (JSON or CSV)
- Highlight high-risk events for manual review

**Hints**
- Reuse pandas to aggregate audit logs
- Tag overrides separately for auditor visibility
- Provide human-readable and machine-readable outputs

In [ ]:
def compile_compliance_report(audit_log: List[Dict[str, Any]]) -> Dict[str, Any]:
    # TODO: Convert audit_log into DataFrame and compute summaries
    ...

def export_report(report: Dict[str, Any], path: str) -> None:
    # TODO: Persist report as JSON/CSV; include timestamp in filename
    ...

def highlight_high_risk_events(steps: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    # TODO: Filter for severity=='critical' or repeated overrides
    ...

# TODO: Generate a sample report from the GuardrailPipeline audit log

**Reflection & Verification**
- [ ] Report covers PII removals, violations, overrides, and trends
- [ ] Auditors can trace high-risk events via correlation IDs
- [ ] Output format aligns with enterprise governance requirements

## Wrap-Up
You now have a modular guardrail system that screens inputs, validates outputs, enforces business policy, and generates compliance artifacts. Continue by:
- Integrating the pipeline with your LLM serving layer
- Shipping timing metrics into observability dashboards for transparency
- Extending rules for prompt injection mitigation (see Lesson 4)
- Running red-team exercises to validate guardrail effectiveness

## Submission Checklist
- [ ] All `TODO` blocks addressed with working implementations
- [ ] Guardrail pipeline demo executed end-to-end with sample inputs
- [ ] Compliance report exported and validated
- [ ] Notes on tuning thresholds, overrides, and future enhancements